In [0]:
from pyspark.sql.functions import (
    col, first, avg, sum as spark_sum, count
)


class GoldAggregation:

    def __init__(self, spark, silver_table, gold_table):
        self.spark = spark
        self.silver_table = silver_table
        self.gold_table = gold_table
        self.df = None
        self.gold_df = None

    def read_silver(self):
        self.df = self.spark.table(self.silver_table)

    def aggregate(self):
        self.gold_df = (
            self.df.groupBy("transaction_date", "product_name", "destination_city")
            .agg(
                # first("product_category").alias("product_category"),
                # first("quantity_unit").alias("quantity_unit"),
            
                avg("supplier_reliability_score").alias("avg_supplier_reliability_score"),
                spark_sum("ordered_quantity").alias("total_ordered_quantity"),
                spark_sum("demand_quantity").alias("total_demand_quantity"),
                spark_sum("available_inventory").alias("total_available_inventory"),
                avg("unit_price_usd").alias("avg_unit_price_usd"),
                spark_sum("product_cost_usd").alias("total_product_cost_usd"),
                spark_sum("transportation_cost_usd").alias("total_transportation_cost_usd"),
                spark_sum("total_cost_usd").alias("total_cost_usd"),
                spark_sum("expected_lead_time_days").alias("total_expected_lead_time_days"),
                spark_sum("actual_lead_time_days").alias("total_actual_lead_time_days"),
                spark_sum("delay_days").alias("total_delay_days"),
                spark_sum(col("is_delayed").cast("int")).alias("total_is_delayed"),
                spark_sum(col("is_stockout").cast("int")).alias("total_is_stockout"),
                avg("quality_score").alias("avg_quality_score"),

                count("*").alias("transaction_count")
            )
        )

    def write_gold(self):
        self.gold_df.write.format("delta") \
            .mode("overwrite") \
            .saveAsTable(self.gold_table)
        print("Gold table created: " + self.gold_table)

    def run(self):
        self.read_silver()
        self.aggregate()
        self.write_gold()


# ---------------------------
# Notebook usage
# ---------------------------
gold_pipeline = GoldAggregation(
    spark=spark,
    silver_table="oil_gas_mlops.silver.silver_supply_chain",
    gold_table="oil_gas_mlops.gold.gold_supply_chain"
)

gold_pipeline.run()